# 🇨🇲 Cameroon SmartTrip — LLM local + Agent

Ce notebook teste la chaîne :

**Utilisateur → LLM local → Agent SmartTrip → SQL / recherche sémantique / budget → réponse**

Aucune clé API payante n'est nécessaire.

> Le premier téléchargement du modèle Hugging Face nécessite Internet, mais pas de clé API payante.


## 1. Importer le projet GitHub-ready

In [ ]:
from google.colab import files
import zipfile, shutil, sys
from pathlib import Path

print("Importez le ZIP Cameroon-SmartTrip_LLM-Agent_GitHub_Ready.zip")
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith(".zip"))

root = Path("/content/smarttrip")
if root.exists():
    shutil.rmtree(root)
root.mkdir()

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(root)

matches = list(root.rglob("Cameroon-SmartTrip-LLM-Agent"))
PROJECT_DIR = matches[0] if matches else next(p for p in root.iterdir() if p.is_dir())
APP_DIR = PROJECT_DIR / "app"
sys.path.insert(0, str(APP_DIR))
print(PROJECT_DIR)


## 2. Installer le moteur et le LLM local

In [ ]:
!pip -q install -r "$PROJECT_DIR/requirements.txt"
!pip -q install -r "$PROJECT_DIR/requirements_llm_local.txt"


## 3. Test gratuit sans modèle — validation de toute la chaîne

In [ ]:
from llm_agent import run_llm_agent

query = (
    "Nous sommes un couple avec deux enfants, budget 300 000 FCFA, "
    "3 jours à Kribi depuis Douala. Je veux piscine, plage et Wi-Fi, "
    "et voir les chutes de la Lobé."
)

test = run_llm_agent(
    query,
    meal_budget_override=40000,
    backend_mode="mock"
)

print("Backend :", test["backend"])
print("Extraction :", test["llm_extracted"])
print("Hôtel :", test["plan"].get("hotel"))
print("Total :", test["plan"].get("total_fcfa"))
print("Réponse :", test["final_answer"])


## 4. Activer le vrai LLM local

Cette cellule utilise par défaut **Qwen2.5-0.5B-Instruct** via Transformers.

Le modèle est petit pour rester testable dans Colab. Aucun token API n'est facturé.


In [ ]:
import os
os.environ["SMARTTRIP_USE_TRANSFORMERS"] = "1"
os.environ["LOCAL_HF_MODEL"] = "Qwen/Qwen2.5-0.5B-Instruct"

real_result = run_llm_agent(
    query,
    meal_budget_override=40000,
    backend_mode="transformers"
)

print("Backend :", real_result["backend"])
print("\nExtraction LLM :")
print(real_result["llm_extracted"])
print("\nPlan de l'agent :")
print(real_result["plan"].get("hotel"), real_result["plan"].get("total_fcfa"))
print("\nRéponse finale du LLM :")
print(real_result["final_answer"])


## 5. Afficher l'itinéraire

In [ ]:
import pandas as pd
display(pd.DataFrame(real_result["itinerary"]))


## Conclusion

Si la cellule 4 fonctionne, Cameroon SmartTrip utilise réellement :

**LLM local + Agent + SQL + recherche sémantique + optimisation**, sans API payante.
